In [ ]:
from roundel_utils import *
from stqdm import stqdm
data_path = '/workspaces/Roundel-Clinical/roundel/data'
os.environ["CUDA_VISIBLE_DEVICES"]="0"

In [ ]:
patient = '1.3.12.2.1107.5.2.18.41548.30000025120809155168300013409'
image = load_nii(f'{data_path}/image___{patient}.nii.gz')
mask = load_nii(f'{data_path}/masks___{patient}.nii.gz')

In [ ]:

def segment_image(image):
    # Crop and pad the image to correct shape

    st.session_state['model'] = tf.keras.models.load_model('SAX-37.h5', 
                                    compile = False,
                                    custom_objects={"InstanceNormalization":InstanceNormalization,
                                                    "ResizeAndConcatenate":ResizeAndConcatenate})

    target_shape = (256,256)
    mask = []
    for t in range(image.shape[-1]):  
        image_cropped, meta =  crop_pad_image_only(image[..., t], target_shape = target_shape)
        X = z_normalise_image(image_cropped)[np.newaxis,...,np.newaxis]
        pred_mask = sliding_window_inference_3d(
                        st.session_state.model,
                        X,                   # np.ndarray [batch_size,Y,X,Z,1] already preprocessed & normalised
                        patch_size=[256,256,10],
                        overlap=0.5,
                        apply_softmax=False,       # set False as the model already uses softmax activation
                        out_channels=5,    # if known, can be set to avoid dry run
                        tta=False,                 # Whether to use test-time augmentation (flips)
                        plot_tta=False,           # Whether to plot the prediction after each TTA variant (for debugging)
                        scan_id = None,           # used for naming the TTA variant plots
                        time_step_counter=0, # used for naming the TTA variant plots
                        gaussian_sigma_scale=1/8, # controls how peaked the Gaussian is
                        deep_supervision=True,
                        run = None               # neptune run instance for logging
                    )
        pred_mask = reverse_crop_pad(pred_mask, meta)

        mask.append(pred_mask)
    mask = np.stack(mask, -1)
    mask = postprocess(mask)
    return mask

In [ ]:
mask = segment_image(image)

In [ ]:
%matplotlib inline
plt.imshow(mask[:,:,6,0])
plt.colorbar()
plt.show()

In [ ]:

fig, axes = plt.subplots(grid_rows, grid_cols, figsize = (20,5))
axes = axes.ravel()

for i, fmap in enumerate(feature_maps):
    fmap = fmap[0]                           # remove batch
    fmap = fmap[:, :, fmap.shape[2]//2, 6]   # middle depth, first channel

    axes[i].imshow(fmap, 'gray')
    axes[i].set_title(layer_names[i], fontsize = 9)
    axes[i].axis("off")

plt.subplots_adjust(wspace = 0.01, left = 0, right = 1, bottom=0, top = 0.98, hspace=0.15)
plt.show()

In [ ]:
t = 5
s = 5
c = 1
plt.imshow(image[...,s,t], 'gray')
plt.imshow(mask[...,s, t, c], alpha = mask[...,s, t, c]/2)